# SharkByte verification on Google Colab

Use a CPU runtime. This notebook runs pinned SharkByte source, creates separate
Python environments for the two reference engines, and exports the JSON
receipts. Windows and Ubuntu ARM64 results come from their own machines.
The 120-circuit run uses Qiskit and Cirq. The 40-qubit run compares every
populated SharkByte amplitude with the factorized Cirq reference.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import tempfile

revision = "bfad672e3cd6b5dbe1e69ac18c36001700244a1f"
run_dir = Path(tempfile.mkdtemp(prefix="sharkbyte-colab-"))
repo = run_dir / "SharkByte"
results = run_dir / "receipts"
results.mkdir()

def run(*args):
    env = dict(os.environ, MPLBACKEND="Agg")
    result = subprocess.run([str(arg) for arg in args], cwd=repo, env=env,
                            capture_output=True, text=True)
    print(result.stdout, end="")
    print(result.stderr, end="", file=sys.stderr)
    result.check_returncode()

subprocess.run(["git", "clone", "https://github.com/reggaesharkk/SharkByte.git", str(repo)], check=True)
run("git", "checkout", "--detach", revision)
print(platform.platform(), platform.machine(), sys.version.split()[0])
print("Results:", results)

In [ ]:
def run_setup(*args):
    command = [str(arg) for arg in args]
    completed = subprocess.run(command, cwd=repo, capture_output=True, text=True)
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    completed.check_returncode()

def install_env(name, *requirements):
    # A fresh path allows this cell to be retried after partial setup.
    location = Path(tempfile.mkdtemp(prefix=f"{name}-", dir=run_dir))
    run_setup(sys.executable, "-m", "venv", "--without-pip", location)
    python = location / "bin" / "python"
    pip = (sys.executable, "-m", "pip", "--python", python)
    run_setup(*pip, "install", *requirements)
    run_setup(*pip, "check")
    return python

qiskit_python = install_env("qiskit", "-r", "requirements-independent-quantum.txt")
cirq_python = install_env("cirq", "cirq-core==1.6.1")


In [ ]:
results = Path(tempfile.mkdtemp(prefix="receipts-", dir=run_dir))
qiskit_receipt = results / "qiskit.json"
run(qiskit_python, "-B", "tools/run_independent_quantum_validation.py",
    "--output", qiskit_receipt)
run(qiskit_python, "-B", "-m", "unittest", "validation.independent_qiskit.test_suite", "-v")
run(cirq_python, "-B", "tools/run_cirq_crosscheck.py",
    qiskit_receipt, "--output", results / "cirq-120.json")

In [ ]:
# This is the large CPU and RAM run. A Colab resource limit can interrupt it.
run(cirq_python, "-B", "tools/run_cirq_scaling_check.py",
    "--output", results / "cirq-16777216.json")

In [ ]:
expected = ["qiskit.json", "cirq-120.json", "cirq-16777216.json"]
summary = {}
for name in expected:
    path = results / name
    if not path.exists():
        summary[name] = {"status": "NO_RECEIPT"}
        continue
    raw = path.read_bytes()
    summary[name] = {"status": json.loads(raw)["status"],
                     "sha256": hashlib.sha256(raw).hexdigest()}

(results / "colab-summary.json").write_text(
    json.dumps({"revision": revision, "host": platform.platform(),
                "architecture": platform.machine(), "checks": summary}, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(summary, indent=2))
archive = shutil.make_archive(str(run_dir / "sharkbyte-colab-evidence"), "zip", results)
from google.colab import files
files.download(archive)